In [10]:
#imports
import requests, zipfile, io, os
from pathlib import Path
import pandas as pd



#methods

def get_base_fn(file_path):
    base_name, _ = os.path.splitext(file_path)
    return base_name

def change_extension(file_path, new_extension):
    # base_name, _ = os.path.splitext(file_path)
    base_name = get_base_fn(file_path)
    new_file_path = base_name + "." + new_extension
    os.rename(file_path, new_file_path)
    print(f"{file_path} is now {new_file_path}" )
    

In [11]:
root = Path()


# create folder inside root, called data
# create folder inside root/data called static
#creating data storage folders in root
data_folder = 'data'
data_pth = root / data_folder
data_pth.mkdir(parents=True, exist_ok=True)
static_folder = 'static'
static_pth = data_pth / static_folder
static_pth.mkdir(parents=True, exist_ok=True)


# download gtfs.zip from https://data.foli.fi/gtfs/gtfs.zip and place inside data.
# unzip data/gtfs.zip into static.
zip_url =  r'https://data.foli.fi/gtfs/gtfs.zip'
r = requests.get(zip_url, stream=True)
z = zipfile.ZipFile(io.BytesIO(r.content))
z.extractall(static_pth)







In [12]:
# rename all files to suffix .csv
for f in static_pth.iterdir():
    if f.name.endswith('.txt'):
        change_extension(f, 'csv')


data/static/stops.txt is now data/static/stops.csv
data/static/trip_notes.txt is now data/static/trip_notes.csv
data/static/agency.txt is now data/static/agency.csv
data/static/stop_times.txt is now data/static/stop_times.csv
data/static/calendar_dates.txt is now data/static/calendar_dates.csv
data/static/trips.txt is now data/static/trips.csv
data/static/calendar.txt is now data/static/calendar.csv
data/static/shapes.txt is now data/static/shapes.csv
data/static/routes.txt is now data/static/routes.csv
data/static/feed_info.txt is now data/static/feed_info.csv
data/static/translations.txt is now data/static/translations.csv


In [35]:
#read all 11 .csv files rand transform into pandas datasets.
all_dfs = {}

for f in static_pth.glob('*.csv'):
    all_dfs[f.stem] = pd.read_csv(f)
#create table names called the filename and column names = column names. 


In [ ]:
for df in all_dfs:
    print(df)
    print(all_dfs[df].columns)
    print(all_dfs[df].head())

calendar_dates
Index(['service_id', 'date', 'exception_type'], dtype='str')
   service_id      date  exception_type
0  1001010100  20260810               1
1  1001010100  20260811               1
2  1001010100  20260812               1
3  1001010100  20260813               1
4  1001010100  20260814               1
stop_times
Index(['trip_id', 'arrival_time', 'departure_time', 'stop_id', 'stop_sequence',
       'stop_headsign', 'pickup_type', 'drop_off_type', 'shape_dist_traveled',
       'timepoint'],
      dtype='str')
                trip_id arrival_time departure_time  stop_id  stop_sequence  \
0  00026078__1039020165     19:19:00       19:19:00     1644              0   
1  00026078__1039020165     19:19:46       19:19:46      449              1   
2  00026078__1039020165     19:20:40       19:20:40      450              2   
3  00026078__1039020165     19:21:20       19:21:20     1645              3   
4  00026078__1039020165     19:22:05       19:22:05      451              4   


'\n-- trips.csv\n-- columns: route_id,service_id,trip_id,trip_headsign,direction_id,block_id,shape_id,wheelchair_accessible,bikes_allowed\n\nCREATE TABLE IF NOT EXISTS trips (\n    trip_id text NOT NULL,\n    route_id INTEGER,\n    service_id INTEGER,\n    trip_headsign text,\n    direction_id SMALLINT,\n    block_id INTEGER,\n    shape_id INTEGER,\n    wheelchair_accessible SMALLINT,\n    bikes_allowed SMALLINT,\n\n    PRIMARY KEY (trip_id),\n    FOREIGN KEY (route_id) REFERENCES routes(route_id)\n);\n\n-- routes.csv\n-- columns: route_id,agency_id,route_short_name,route_long_name,route_desc,route_type,route_url,route_color,route_text_color\nCREATE TABLE IF NOT EXISTS routes (\n    route_id INTEGER NOT NULL,\n    agency_id INTEGER NOT NULL,\n    route_short_name VARCHAR(5),\n    route_long_name TEXT,\n    route_desc TEXT,\n    route_type NUM,\n    route_url TEXT,\n    route_color VARCHAR(6),\n    route_text_color VARCHAR(6),\n\n    FOREIGN KEY (agency_id) REFERENCES agency(agency_id)\

In [ ]:
big_fat_sql_command = """
-- trips.csv
-- columns: route_id,service_id,trip_id,trip_headsign,direction_id,block_id,shape_id,wheelchair_accessible,bikes_allowed

CREATE TABLE IF NOT EXISTS trips (
    trip_id text NOT NULL,
    route_id TEXT,
    service_id TEXT,
    trip_headsign text,
    direction_id SMALLINT,
    block_id INTEGER,
    shape_id TEXT,
    wheelchair_accessible SMALLINT,
    bikes_allowed SMALLINT,
    
    PRIMARY KEY (trip_id),
    FOREIGN KEY (route_id) REFERENCES routes(route_id),

);

-- routes.csv
-- columns: route_id,agency_id,route_short_name,route_long_name,route_desc,route_type,route_url,route_color,route_text_color
CREATE TABLE IF NOT EXISTS routes (
    route_id TEXT NOT NULL,
    agency_id TEXT NOT NULL,
    route_short_name VARCHAR(20),
    route_long_name TEXT,
    route_desc TEXT,
    route_type INTEGER,
    route_url TEXT,
    route_color VARCHAR(6),
    route_text_color VARCHAR(6),

    PRIMARY KEY (route_id),
    FOREIGN KEY (agency_id) REFERENCES agency(agency_id)

);

-- agency.csv
-- columns: agency_id,agency_name,agency_url,agency_timezone,agency_lang,agency_phone,agency_fare_url
-- example: 100,"V-S ELY-keskus, palveluntuottaja FinFerries",https://www.finferries.fi/,Europe/Helsinki,fi,0207 118 750,

CREATE TABLE IF NOT EXISTS agency(
    agency_id TEXT NOT NULL,
    agency_name TEXT,
    agency_url TEXT,
    agency_timezone TEXT,
    agency_lang VARCHAR(5),
    agency_phone VARCHAR(24),
    agency_fare_url TEXT,

    PRIMARY KEY (agency_id)

);



-- trip_notes.csv
-- columns: trip_id,abbreviation,description,lang
CREATE TABLE IF NOT EXISTS trip_notes (
    trip_id text NOT NULL, 
    abbreviation text, 
    description text, 
    language text NOT NULL,

    PRIMARY KEY (trip_id, language),
    FOREIGN KEY (trip_id) REFERENCES trips(trip_id)
);


-- stops.csv
-- columns:stop_id,stop_code,stop_name,stop_desc,stop_lat,stop_lon,zone_id,stop_url,location_type,parent_station,stop_timezone,wheelchair_boarding,platform_code

CREATE TABLE IF NOT EXISTS stops(
    stop_id TEXT NOT NULL, 
    stop_code TEXT,
    stop_name VARCHAR(100),
    stop_desc text,
    stop_lat DOUBLE PRECISION,
    stop_lon DOUBLE PRECISION,
    zone_id VARCHAR(10),
    stop_url text,
    location_type SMALLINT,
    parent_station text,
    stop_timezone text,
    wheelchair_boarding SMALLINT,
    platform_code VARCHAR(50),

    PRIMARY KEY (stop_id)
);

--stop_times.csv
-- columns: trip_id,arrival_time,departure_time,stop_id,stop_sequence,stop_headsign,pickup_type,drop_off_type,shape_dist_traveled,timepoint
--ex: "00014922__1006060106",06:21:00,06:25:00,1901,32,,0,0,15072,1
-- 1901 is from stops.csv 1901,1901,Kauppatori A1,,60.45168218,22.26545694,FÖLI,,0,,Europe/Helsinki,0,A1


CREATE TABLE IF NOT EXISTS stop_times(
    trip_id text NOT NULL,
    arrival_time VARCHAR(8), -- this and departure_time goes past midnight, will need to parsed post pull from db
    departure_time VARCHAR(8),
    stop_id text,
    stop_sequence INTEGER NOT NULL,
    stop_headsign VARCHAR(150),
    pickup_type INTEGER,
    drop_off_type INTEGER,
    shape_dist_traveled INTEGER,
    timepoint INTEGER,

    PRIMARY KEY (trip_id, stop_sequence),
    FOREIGN KEY (trip_id) REFERENCES trips(trip_id),
    FOREIGN KEY (stop_id) REFERENCES stops(stop_id)

);


-- stops_translations and trip_headsign_translations was one csv, called translations.csv
-- translations.csv not that easy, contains two types of rows
-- columns: table_name,field_name,language,translation,record_id,field_value
-- rowtype1: stops,stop_name,sv,Satava,386,
-- rowtype2: trips,trip_headsign,sv,Lundo-Tarvasjoki-Koskis,,Lieto-Tarvasjoki-Koski Tl

CREATE TABLE IF NOT EXISTS stops_translations(
    stop_id TEXT, --corresponds directly to the original finnish name in stops.csv so foreign key to stop_id 
    language text,
    translation text, -- the translation of stop_id in stops.csv

    PRIMARY KEY (stop_id, language),
    FOREIGN KEY (stop_id) REFERENCES stops(stop_id)
    
);

CREATE TABLE IF NOT EXISTS trip_headsign_translations(
    original TEXT, -- maps to field_value, uncertain atm how to get to this, iguess just a costly search
    language VARCHAR(10),
    translation TEXT,
    
    PRIMARY KEY (original, language)
    
);


-- calendar_dates.csv
-- service_id,date,exception_type
-- 1001010100,20260810,1
-- 1001010100,20260811,1

CREATE TABLE IF NOT EXISTS calendar_dates(
    service_id TEXT NOT NULL,
    date VARCHAR(8), --transform into date in service layer ig
    exception_type INTEGER, 

    PRIMARY KEY (service_id),

);


--skipped - calendar.csv
--service_id,monday,tuesday,wednesday,thursday,friday,saturday,sunday,start_date,end_date
--1001010100,0,0,0,0,0,0,0,20260810,20270606

CREATE TABLE IF NOT EXISTS calendar(
    service_id TEXT NOT NULL,
    monday SMALLINT,
    tuesday SMALLINT,
    wednesday SMALLINT,
    thursday SMALLINT,
    friday SMALLINT,
    saturday SMALLINT,
    sunday SMALLINT,
    start_date VARCHAR(8),
    end_date VARCHAR(8),

    PRIMARY KEY (service_id, date)
    );

"""









#skipped - feed_info.csv
#feed_publisher_name,feed_publisher_url,feed_lang
#FÖLI Turun seudun joukkoliikenne,https://www.foli.fi/fi,fi
#(all it contains)

#skipped - shapes.csv 
# shape_id,shape_pt_lat,shape_pt_lon,shape_pt_sequence,shape_dist_traveled
# 175,60.3956367,22.1681544,0,0
# 175,60.395722,22.168318,1,13
#for now, we need to get this later.